In [23]:
from pathlib import Path
import json
import PyPDF2

SAVE_PATH = Path('../data/corpus/documents')
PDF_PATH = SAVE_PATH / 'pnns_5_2026-2030.pdf'

SAVE_PATH.mkdir(parents=True, exist_ok=True)

with open(PDF_PATH, 'rb') as file:
    reader = PyPDF2.PdfReader(file)
    print(f'Nombre de pages : {len(reader.pages)}')

    complet_text = ""

    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        complet_text += text + '\n'     

Nombre de pages : 72


In [24]:
def chunk_with_overlap(
        text: str, 
        chunk_size: int = 500, 
        overlap: int = 50
) -> list[str]:
    start = 0
    result = []

    while start < len(text):
        chunk = text[start : start + chunk_size]
        result.append(chunk)
        start += (chunk_size - overlap)

    return result

chunks = chunk_with_overlap(complet_text, 500, 50)

print(chunks[3])

climat, qui porte une 
vision claire : concilier santé humaine, justice 
sociale et transition écologique. Car manger 
mieux, bouger plus, protéger notre environ-
nement et réduire les inégalités ne sont pas 
des objectifs isolés mais les fondations d’un 
progrès véritablement durable.Le 5ème PNNS réaffirme ainsi la place 
centrale de l’alimentation et de l’activité 
physique comme déterminant majeur de 
santé. Mieux informer, mieux accompa-
gner et rendre plus accessibles des choix 
alimentaire


In [25]:
def transform_chunks(chunks: list):
    documents = []

    for i, chunk in enumerate(chunks):
        doc = {
            'id': f'pnns:chunk_{i}',
            'texte': chunk,
            'metadonnees': {
                'source': 'PNNS 2026-2030',
                'licence': 'ODbL'
            }
        }

        documents.append(doc)

    return documents 

documents = transform_chunks(chunks) 
print(documents[0]) 

{'id': 'pnns:chunk_0', 'texte': 'Programme\nNational\nNutrition\nSanté\n2026 - 2030\nProgramme National Nutrition Santé | 2026 -20302\nÉditorial\nStéphanie RIST\nMinistre de la Santé, des Familles, de \nl’Autonomie et des Personnes handicapées\nDepuis plus de vingt ans, notre pays met \nen œuvre une politique nationale pour \naméliorer durablement la santé de la popu -\nlation à travers une meilleure alimentation \net une augmentation de la pratique d’acti-\nvité physique. Ce 5ème Programme national \nnutrition santé 2025-2030 vise à répondre ', 'metadonnees': {'source': 'PNNS 2026-2030', 'licence': 'ODbL'}}


In [28]:
def save_chunks_to_textfile(documents: list[str]):
    for document in documents:
        path = SAVE_PATH / f'{document['id'].replace(':', '_')}.txt'
        path.write_text(document['texte'], encoding='utf-8')

    path_meta = SAVE_PATH / 'metadonnees.jsonl'
    with open(path_meta, 'w', encoding='utf-8') as file:
        for document in documents:
            row = json.dumps({
                'id': document['id'],
                'source': document['metadonnees']['source'],
                'licence': document['metadonnees']['licence']
            })

            file.write(row + '\n')

save_chunks_to_textfile(documents)